## Результаты алгоритма с реранкером

In [1]:
import polars as pl

In [2]:
golden_set = pl.read_parquet("../data/golden_set.parquet").filter(pl.col("geonameIds").list.len() > 0)

In [3]:
import httpx
from tqdm.auto import tqdm


def search_batch(queries: list[str], top_k: int = 50, use_rerank: bool = True):
    base_url = "http://localhost:8000/v1/search"
    results = []
    
    with httpx.Client(timeout=30.0) as client:  # синхронный клиент
        for query in tqdm(queries):
            response = client.get(base_url, params={"text": query, "top_k": top_k, "use_rerank": use_rerank})
            response.raise_for_status()
            results.append(response.json())
    
    return results

In [4]:
from ir_measures import P, Recall, RR, calc


qrels_dict = {}
for row in golden_set.iter_rows(named=True):
    qrels_dict.update({row["query"]: {str(gid): 1 for gid in row["geonameIds"]}})

predictions = search_batch(qrels_dict.keys(), top_k=50, use_rerank=True)

  0%|          | 0/180 [00:00<?, ?it/s]

In [5]:
run_dict = {
    pred["query"]: {
        str(r["geonameid"]): len(pred["results"]) - i
        for i, r in enumerate(pred["results"])
    }
    for pred in predictions
}

metrics = calc([RR, P@1, Recall@5, Recall@10, Recall@25, Recall@50], qrels_dict, run_dict)
metrics_aggregated = pl.DataFrame({str(k): v for k, v in metrics.aggregated.items()}).unpivot().sort("value")

metrics_per_query = {str(m): {"query": [], "value": []} for m in metrics.aggregated}

for metric in metrics.per_query:
    mname = str(metric.measure)
    metrics_per_query[mname]["query"].append(metric.query_id)
    metrics_per_query[mname]["value"].append(metric.value)

for mname in metrics_per_query:
    metrics_per_query[mname] = pl.DataFrame(metrics_per_query[mname]).sort("value")

In [6]:
metrics_aggregated

variable,value
str,f64
"""P@1""",0.883333
"""R@5""",0.894004
"""RR""",0.914709
"""R@10""",0.923913
"""R@25""",0.95428
"""R@50""",0.95782


In [7]:
for k in metrics_per_query:
    print(k)
    display(metrics_per_query[k].limit(5))

RR


query,value
str,f64
"""Ağrı'da kış mevsiminde turist …",0.0
"""Van'da bir hafta sonu tatili i…",0.0
"""Merhaba, İstanbul'un Fatih sem…",0.0
"""St. Louis'de Gateway Kemeri'ni…",0.047619
"""Какие достопримечательности ст…",0.083333


R@25


query,value
str,f64
"""Ağrı'da kış mevsiminde turist …",0.0
"""Van'da bir hafta sonu tatili i…",0.0
"""Merhaba, İstanbul'un Fatih sem…",0.0
"""Is Omsk or Yaroslavl a better …",0.5
"""Van, Sinop ve Şanlıurfa’dan bi…",0.5


P@1


...

query,value
str,f64
"""Ağrı'da kış mevsiminde turist …",0.0
"""Какие достопримечательности ст…",0.0
"""St. Louis'de Gateway Kemeri'ni…",0.0
"""Где в Туле можно поесть настоя…",0.0
"""Где лучше жить: в Омахе или в …",0.0


R@50


query,value
str,f64
"""Ağrı'da kış mevsiminde turist …",0.0
"""Van'da bir hafta sonu tatili i…",0.0
"""Merhaba, İstanbul'un Fatih sem…",0.0
"""Is Omsk or Yaroslavl a better …",0.5
"""Van, Sinop ve Şanlıurfa’dan bi…",0.5


R@10


query,value
str,f64
"""Ağrı'da kış mevsiminde turist …",0.0
"""Какие достопримечательности ст…",0.0
"""St. Louis'de Gateway Kemeri'ni…",0.0
"""Где лучше жить: в Омахе или в …",0.0
"""Van'da bir hafta sonu tatili i…",0.0
